# Recommendation System Project: IBM Community

In this notebook, you will be putting your recommendation skills to use on real data from the IBM Watson Studio platform. 


You may either submit your notebook through the workspace here, or you may work from your local machine and submit through the next page.  Either way assure that your code passes the project rubric (see the **Project Rubric** page in the classroom).  **Please save regularly.**

By following the table of contents, you will build out a number of different methods for making recommendations that can be used for different situations. 


## Table of Contents

I. [Exploratory Data Analysis](#Exploratory-Data-Analysis)<br>
II. [Rank Based Recommendations](#Rank)<br>
III. [User-User Based Collaborative Filtering](#User-User)<br>
IV. [Content Based Recommendations](#Content-Recs)<br>
V. [Matrix Factorization](#Matrix-Fact)<br>
VI. [Extras & Concluding](#conclusions)

At the end of the notebook, you will find directions for how to submit your work.  Let's get started by importing the necessary libraries and reading in the data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import project_tests as t


df = pd.read_csv(
    'data/user-item-interactions.csv', 
    dtype={'article_id': int, 'title': str, 'email': str}
)
# Show df to get an idea of the data
df.head()

,Unnamed: 0,article_id,title,email
0,0,1430,"using pixiedust for fast, flexible, and easier...",ef5f11f77ba020cd36e1105a00ab868bbdbf7fe7
1,1,1314,healthcare python streaming application demo,083cbdfa93c8444beaa4c5f5e0f5f9198e4f9e0b
2,2,1429,use deep learning for image classification,b96a4f2e92d8572034b1e9b28f9ac673765cd074
3,3,1338,ml optimization using cognitive assistant,06485706b34a5c9bf2a0ecdac41daf7e7654ceb7
4,4,1276,deploy your python model as a restful api,f01220c46fc92c6e6b161b1849de11faacd7ccb2


### <a class="anchor" id="Exploratory-Data-Analysis">Part I : Exploratory Data Analysis</a>

Use the dictionary and cells below to provide some insight into the descriptive statistics of the data.

`1.` Are there any missing values? If so, provide a count of missing values. If there are missing values in `email`, assign it the same id value `"unknown_user"`.

In [2]:
# Some interactions do not have a user associated with it, assume the same user.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45993 entries, 0 to 45992
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  45993 non-null  int64
 1   article_id  45993 non-null  int64
 2   title       45993 non-null  str  
 3   email       45976 non-null  str  
dtypes: int64(2), str(2)
memory usage: 1.4 MB


In [3]:
print(f"Number of Null email values is: {df.email.isna().sum()}")

Number of Null email values is: 17


In [4]:
df[df.email.isna()]

,Unnamed: 0,article_id,title,email
25131,25146,1016,why you should master r (even if it might even...,NaN
29758,30157,1393,the nurse assignment problem,NaN
29759,30158,20,working interactively with rstudio and noteboo...,NaN
29760,30159,1174,breast cancer wisconsin (diagnostic) data set,NaN
29761,30160,62,data visualization: the importance of excludin...,NaN
35264,36016,224,"using apply, sapply, lapply in r",NaN
35276,36029,961,beyond parallelize and collect,NaN
35277,36030,268,sector correlations shiny app,NaN
35278,36031,268,sector correlations shiny app,NaN
35279,36032,268,sector correlations shiny app,NaN


In [5]:
# Fill email NaNs with "unknown_user"
df["email"] = df["email"].fillna("unknown_user")

In [6]:
# Check if no more NaNs 
df[df.email.isna()]

,Unnamed: 0,article_id,title,email


`2.` What is the distribution of how many articles a user interacts with in the dataset?  Provide a visual and descriptive statistics to assist with giving a look at the number of times each user interacts with an article.

In [7]:
# What are the descriptive statistics of the number of articles a user interacts with?
# Each row is one interaction, so repeat views of the same article count
# again, which is how the course's own tests count them.
interactions_per_user = df.groupby("email").size()
interactions_per_user.describe()

count    5149.000000
mean        8.932414
std        16.801011
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       364.000000
dtype: float64

In [8]:
# Create a plot of the number of articles read by each user
plt.hist(interactions_per_user, bins=50, log=True)
plt.xlabel('number of articles')
plt.ylabel('number of users')
plt.title('Number of Users Reading Articles')
plt.show()

/var/folders/mq/m00xc0l165lcfkh2tz919jdh0000gn/T/ipykernel_6825/2203500280.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Create a plot of the number of times each article was read
views_per_article = df.groupby("article_id").size()
plt.hist(views_per_article, bins=50, log=True)
plt.xlabel('number of users')
plt.ylabel('number of articles')
plt.title('Distribution of Article Usage')
plt.show()

/var/folders/mq/m00xc0l165lcfkh2tz919jdh0000gn/T/ipykernel_6825/1830039096.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Fill in the median and maximum number of user_article interactions below

median_val = int(interactions_per_user.median())  # 50% of individuals interact with ____ number of articles or fewer.
max_views_by_user = int(interactions_per_user.max())  # The maximum number of user-article interactions by any 1 user is ______.

`3.` Use the cells below to find:

**a.** The number of unique articles that have an interaction with a user.  
**b.** The number of unique articles in the dataset.<br>
**c.** The number of unique users in the dataset. (excluding null values) <br>
**d.** The number of user-article interactions in the dataset.

In [11]:
unique_articles = int(df["article_id"].nunique())  # The number of unique articles that have at least one interaction
total_articles = int(df["article_id"].nunique())  # The number of unique articles on the IBM platform
unique_users = int(df["email"].nunique())  # The number of unique users
user_article_interactions = int(len(df))  # The number of user-article interactions

**How these four numbers are counted.** This version of the project has only the interactions file (there is no separate article catalogue), so the number of articles on the platform and the number with at least one interaction are the same count. The user count includes the one `unknown_user` that the missing emails were mapped to in question 1; the course's test for this dictionary expects that user to be counted. The cell below prints both counts so the difference is visible, and shows how many rows are a user returning to an article they had already opened.

In [12]:
known_users = df.loc[df["email"] != "unknown_user", "email"].nunique()
unique_pairs = len(df.drop_duplicates(["email", "article_id"]))
print(f"Users including unknown_user: {unique_users}")
print(f"Users excluding unknown_user: {known_users}")
print(f"Distinct user-article pairs: {unique_pairs}")
print(f"Repeat views (rows beyond the first per pair): {len(df) - unique_pairs}")

Users including unknown_user: 5149
Users excluding unknown_user: 5148
Distinct user-article pairs: 33682
Repeat views (rows beyond the first per pair): 12311


`4.` Use the cells below to find the most viewed **article_id**, as well as how often it was viewed.  After talking to the company leaders, the `email_mapper` function was deemed a reasonable way to map users to ids.  There were a small number of null values, and it was found that all of these null values likely belonged to a single user (which is how they are stored using the function below).

In [13]:
views_by_article = df["article_id"].value_counts()
most_viewed_article_id = int(views_by_article.idxmax())  # The article_id of the most viewed article in the dataset, as an integer
max_views = int(views_by_article.max())  # The most viewed article in the dataset was viewed how many times?

In [14]:
## No need to change the code here - this will be helpful for later parts of the notebook
# Run this cell to map the user email to a user_id column and remove the email column

def email_mapper(df=df):
    """Map each email to a whole-number user id, in order of first appearance.

    INPUT:
    df - (pandas dataframe) df as defined at the top of the notebook, with
         the missing emails already filled with "unknown_user"

    OUTPUT:
    user_ids - (list) one user id per row of df, starting at 1; every row
               with the same email gets the same id
    """
    coded_dict = {
        email: num 
        for num, email in enumerate(df['email'].unique(), start=1)
    }
    return [coded_dict[val] for val in df['email']]

df['user_id'] = email_mapper(df)
del df['email']

# show header
df.head()

,Unnamed: 0,article_id,title,user_id
0,0,1430,"using pixiedust for fast, flexible, and easier...",1
1,1,1314,healthcare python streaming application demo,2
2,2,1429,use deep learning for image classification,3
3,3,1338,ml optimization using cognitive assistant,4
4,4,1276,deploy your python model as a restful api,5


In [15]:
## If you stored all your results in the variable names above, 
## you shouldn't need to change anything in this cell

sol_1_dict = {
    '`50% of individuals have _____ or fewer interactions.`': median_val,
    '`The total number of user-article interactions in the dataset is ______.`': user_article_interactions,
    '`The maximum number of user-article interactions by any 1 user is ______.`': max_views_by_user,
    '`The most viewed article in the dataset was viewed _____ times.`': max_views,
    '`The article_id of the most viewed article is ______.`': most_viewed_article_id,
    '`The number of unique articles that have at least 1 rating ______.`': unique_articles,
    '`The number of unique users in the dataset is ______`': unique_users,
    '`The number of unique articles on the IBM platform`': total_articles
}

# Test your dictionary against the solution
t.sol_1_test(sol_1_dict)

It looks like you have everything right here! Nice job!


### <a class="anchor" id="Rank">Part II: Rank-Based Recommendations</a>

In this project, we don't actually have ratings for whether a user liked an article or not.  We only know that a user has interacted with an article. In these cases, the popularity of an article can really only be based on how often an article was interacted with.

`1.` Fill in the function below to return the **n** top articles ordered with most interactions as the top. Test your function using the tests below.

In [16]:
def _article_view_ranking(df=df):
    """Rank every article by its number of interactions (rows), most first.

    Ties are broken by article id, smallest first, so the order is the same
    on every run and every machine.

    INPUT:
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    ranking - (pandas dataframe) columns article_id, title, views; one row
              per article, sorted by views descending
    """
    views = (
        df.groupby(["article_id", "title"])
        .size()
        .rename("views")
        .reset_index()
    )
    return views.sort_values(
        ["views", "article_id"], ascending=[False, True]
    ).reset_index(drop=True)


def get_top_articles(n, df=df):
    """
    INPUT:
    n - (int) the number of top articles to return
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    top_articles - (list) A list of the top 'n' article titles

    """
    top_articles = _article_view_ranking(df)["title"].head(n).tolist()

    return top_articles # Return the top article titles from df

def get_top_article_ids(n, df=df):
    """
    INPUT:
    n - (int) the number of top articles to return
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    top_articles - (list) A list of the top 'n' article ids, as ints

    """
    top_articles = [int(a) for a in _article_view_ranking(df)["article_id"].head(n)]

    return top_articles # Return the top article ids

In [17]:
print(get_top_articles(10))
print(get_top_article_ids(10))

['use deep learning for image classification', 'insights from new york car accident reports', 'visualize car data with brunel', 'use xgboost, scikit-learn & ibm watson machine learning apis', 'predicting churn with the spss random tree algorithm', 'healthcare python streaming application demo', 'finding optimal locations of new store using decision optimization', 'apache spark lab, part 1: basic concepts', 'analyze energy consumption in buildings', 'gosales transactions for logistic regression model']
[1429, 1330, 1431, 1427, 1364, 1314, 1293, 1170, 1162, 1304]


In [18]:
# Test your function by returning the top 5, 10, and 20 articles
top_5 = get_top_articles(5)
top_10 = get_top_articles(10)
top_20 = get_top_articles(20)

# Test each of your three lists from above
t.sol_2_test(get_top_articles)

Your top_5 looks like the solution list! Nice job.
Your top_10 looks like the solution list! Nice job.
Your top_20 looks like the solution list! Nice job.


**Views are not readers.** The ranking above counts every view, as the course's test expects. One reader opening an article many times counts as many views, so the cell below lists the top ten by views beside the number of different users who read each one, and the largest share of an article's views that came from a single user.

In [19]:
top10 = _article_view_ranking(df).head(10).copy()
readers = df.groupby("article_id")["user_id"].nunique()
heaviest = df.groupby(["article_id", "user_id"]).size().groupby("article_id").max()
top10["readers"] = top10["article_id"].map(readers)
top10["heaviest_user_views"] = top10["article_id"].map(heaviest)
top10["reader_rank"] = top10["article_id"].map(
    readers.rank(method="min", ascending=False).astype(int)
)
print(top10[["article_id", "views", "readers", "reader_rank", "heaviest_user_views"]].to_string(index=False))
print(f"Most-viewed article: {int(top10['article_id'].iloc[0])} ({int(top10['views'].iloc[0])} views)")
print(f"Most-read article by distinct readers: {int(readers.idxmax())} ({int(readers.max())} readers)")

 article_id  views  readers  reader_rank  heaviest_user_views
       1429    937      397            2                   35
       1330    927      467            1                   14
       1431    671      320            6                   22
       1427    643      308            8                   22
       1364    627      388            3                   16
       1314    614      345            4                   11
       1293    572      225           28                   16
       1170    565      279           16                   42
       1162    512      245           22                   11
       1304    483      282           13                   14
Most-viewed article: 1429 (937 views)
Most-read article by distinct readers: 1330 (467 readers)


### <a class="anchor" id="User-User">Part III: User-User Based Collaborative Filtering</a>


`1.` Use the function below to reformat the **df** dataframe to be shaped with users as the rows and articles as the columns.  

* Each **user** should only appear in each **row** once.


* Each **article** should only show up in one **column**.  


* **If a user has interacted with an article, then place a 1 where the user-row meets for that article-column**.  It does not matter how many times a user has interacted with the article, all entries where a user has interacted with an article should be a 1.  


* **If a user has not interacted with an item, then place a zero where the user-row meets for that article-column**. 

Use the tests to make sure the basic structure of your matrix matches what is expected by the solution.

In [20]:
# create the user-article matrix with 1's and 0's

def create_user_item_matrix(df, fill_value=0):
    """
    INPUT:
    df - pandas dataframe with article_id, title, user_id columns
    fill_value - value used for user-article pairs with no interaction
                 before the matrix is turned into 1's and 0's

    OUTPUT:
    user_item - user item matrix

    Description:
    Return a matrix with user ids as rows and article ids on the columns with 1 values where a user interacted with
    an article and a 0 otherwise
    """
    # Count interactions per user-article pair, then keep only whether
    # any interaction happened: repeat views still give a single 1.
    counts = (
        df.groupby(["user_id", "article_id"])
        .size()
        .unstack(fill_value=fill_value)
    )
    user_item = (counts > 0).astype(int)

    return user_item # return the user_item matrix

user_item = create_user_item_matrix(df)

In [21]:
## Tests: You should just need to run this cell.  Don't change the code.
assert user_item.shape[0] == 5149, "Oops!  The number of users in the user-article matrix doesn't look right."
assert user_item.shape[1] == 714, "Oops!  The number of articles in the user-article matrix doesn't look right."
assert user_item.sum(axis=1)[1] == 36, "Oops!  The number of articles seen by user 1 doesn't look right."
print("You have passed our quick tests!  Please proceed!")

You have passed our quick tests!  Please proceed!


`2.` Complete the function below which should take a user_id and provide an ordered list of the most similar users to that user (from most similar to least similar).  The returned result should not contain the provided user_id, as we know that each user is similar to him/herself. Because the results for each user here are binary, it (perhaps) makes sense to compute similarity as the dot product of two users. 

Use the tests to test your function.

In [22]:
# Lets use the cosine_similarity function from sklearn
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
def find_similar_users(user_id, user_item=user_item, include_similarity=False):
    """
    INPUT:
    user_id - (int) a user_id
    user_item - (pandas dataframe) matrix of users by articles:
                1's when a user has interacted with an article, 0 otherwise
    include_similarity - (bool) whether to include the similarity in the output

    OUTPUT:
    similar_users - (list) an ordered list where the closest users (largest
                    cosine similarity) are listed first

    Description:
    Computes the cosine similarity of the given user to every other user.
    For 0/1 rows this is the dot product (the number of shared articles)
    divided by the square root of the product of the two users' article
    counts. Both of those are whole numbers, so the dot product is exact and
    only one square root and one division are rounded. IEEE 754 rounds both
    exactly, so the same similarities come out on every machine, which a
    library matrix product does not promise.
    Returns an ordered list of user ids. If include_similarity is True, returns a list of lists
    where the first element is the user id and the second the similarity.
    Users with the same similarity are listed by user id, smallest first.

    """
    matrix = user_item.to_numpy(dtype=np.int64)
    target = user_item.loc[user_id].to_numpy(dtype=np.int64)
    # compute similarity of each user to the provided user
    shared = matrix @ target
    sizes = matrix.sum(axis=1)
    similarity = shared / np.sqrt(sizes * int(target.sum()))

    # sort by similarity
    ranked = pd.DataFrame({"user_id": user_item.index, "similarity": similarity})
    ranked = ranked.sort_values(
        ["similarity", "user_id"], ascending=[False, True], kind="mergesort"
    )
    # remove the own user's id
    ranked = ranked[ranked["user_id"] != user_id]

    # create list of just the ids
    most_similar_users = [int(u) for u in ranked["user_id"]]

    # create list of just the similarities
    similarities = [float(s) for s in ranked["similarity"]]

    if include_similarity:
        return [[u, s] for u, s in zip(most_similar_users, similarities)] # return a list of lists with [[similar_user_id, similarity], ...]
    return most_similar_users # return a list of the users in order from most to least similar
        

In [24]:
# Do a spot check of your function
print("The 10 most similar users to user 1 are: {}".format(find_similar_users(1)[:10]))
print("The 5 most similar users to user 3933 are: {}".format(find_similar_users(3933)[:5]))
print("The 3 most similar users to user 46 are: {}".format(find_similar_users(46)[:3]))

The 10 most similar users to user 1 are: [3933, 46, 4201, 253, 824, 5034, 5041, 136, 2305, 395]
The 5 most similar users to user 3933 are: [1, 46, 4201, 253, 824]
The 3 most similar users to user 46 are: [4201, 790, 5077]


`3.` Now that you have a function that provides the most similar users to each user, you will want to use these users to find articles you can recommend.  Complete the functions below to return the articles you would recommend to each user. 

In [25]:
def get_article_names(article_ids, df=df):
    """
    INPUT:
    article_ids - (list) a list of article ids
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    article_names - (list) a list of article names associated with the list of article ids
                    (this is identified by the title column in df)
                    in the same order as article_ids
    """
    # Each article id has exactly one title in this data.
    titles = df.drop_duplicates("article_id").set_index("article_id")["title"]
    article_names = [titles[int(a)] for a in article_ids]

    return article_names # Return the article names associated with list of article ids

def get_ranked_article_unique_counts(article_ids, user_item=user_item):
    """
    INPUT:
    article_ids - (list) a list of article ids to evaluate (a subset to rank).
                the dtype should match the column labels in `user_item`.
    user_item - (pandas dataframe) a binary matrix of users by articles [n_users x n_articles]:
                1's when a user has interacted with an article, 0 otherwise
                columns are article IDs.

    OUTPUT:
    ranked_article_unique_counts - (list) a list of [article_id, unique_user_count]
                     pairs, only for the provided `article_ids`, sorted by the
                     number of unique users in descending order (ties by
                     article id, smallest first). Each pair is a two-item
                     list rather than a tuple, because the course's test
                     compares the first pair with a list.

    Description:
    For each `article_id` in the input list, count the number of unique users who
    interacted with it (column sum in the binary `user_item` matrix).
    Return the (article_id, count) pairs sorted in descending order.
    """
    counts = user_item[list(article_ids)].sum(axis=0)
    pairs = sorted(
        ([int(a), int(c)] for a, c in counts.items()),
        key=lambda pair: (-pair[1], pair[0]),
    )
    ranked_article_unique_counts = pairs

    return ranked_article_unique_counts


def get_user_articles(user_id, user_item=user_item):
    """
    INPUT:
    user_id - (int) a user id
    user_item - (pandas dataframe) matrix of users by articles:
                1's when a user has interacted with an article, 0 otherwise

    OUTPUT:
    article_ids - (list) a list of the article ids seen by the user
    article_names - (list) a list of article names associated with the list of article ids
                    (this is identified by the title column in df)

    Description:
    Provides a list of the article_ids and article titles that have been seen by a user
    """
    row = user_item.loc[user_id]
    article_ids = [int(a) for a in row[row == 1].index]
    article_names = get_article_names(article_ids)

    return article_ids, article_names # return the ids and names


def user_user_recs(user_id, m=10):
    """
    INPUT:
    user_id - (int) a user id
    m - (int) the number of recommendations you want for the user

    OUTPUT:
    recs - (list) a list of recommendations for the user

    Description:
    Loops through the users based on closeness to the input user_id
    For each user - finds articles the user hasn't seen before and provides them as recs
    Does this until m recommendations are found

    Notes:
    Users who are the same closeness are chosen arbitrarily as the 'next' user

    For the user where the number of recommended articles starts below m
    and ends exceeding m, the last items are chosen arbitrarily

    """
    seen = set(get_user_articles(user_id)[0])
    recs = []
    for neighbor in find_similar_users(user_id):
        for article in get_user_articles(neighbor)[0]:
            if article not in seen and article not in recs:
                recs.append(article)
        if len(recs) >= m:
            break
    recs = recs[:m]

    return recs # return your recommendations for this user_id    

In [26]:
# Check Results
get_article_names(user_user_recs(1, 10)) # Return 10 recommendations for user 1

['this week in data science (april 18, 2017)',
 'this week in data science (may 2, 2017)',
 'top 20 r machine learning and data science packages',
 'improving the roi of big data and analytics through leveraging new sources of data',
 'using apply, sapply, lapply in r',
 'awesome deep learning papers',
 'leverage python, scikit, and text classification for behavioral profiling',
 'challenges in deep learning',
 'do i need to learn r?',
 'how can data scientists collaborate to build better business']

In [27]:
get_ranked_article_unique_counts([1320, 232, 844])

[[1320, 123], [844, 78], [232, 62]]

In [28]:
# Test your functions here - No need to change this code - just run this cell
assert set(get_article_names([1024, 1176, 1305, 1314, 1422, 1427])) == set(['using deep learning to reconstruct high-resolution audio', 'build a python app on the streaming analytics service', 'gosales transactions for naive bayes model', 'healthcare python streaming application demo', 'use r dataframes & ibm watson natural language understanding', 'use xgboost, scikit-learn & ibm watson machine learning apis']), "Oops! Your the get_article_names function doesn't work quite how we expect."
assert set(get_article_names([1320, 232, 844])) == set(['housing (2015): united states demographic measures','self-service data preparation with ibm data refinery','use the cloudant-spark connector in python notebook']), "Oops! Your the get_article_names function doesn't work quite how we expect."
assert set(get_user_articles(20)[0]) == set([1320, 232, 844])
assert set(get_user_articles(20)[1]) == set(['housing (2015): united states demographic measures', 'self-service data preparation with ibm data refinery','use the cloudant-spark connector in python notebook'])
assert set(get_user_articles(2)[0]) == set([1024, 1176, 1305, 1314, 1422, 1427])
assert set(get_user_articles(2)[1]) == set(['using deep learning to reconstruct high-resolution audio', 'build a python app on the streaming analytics service', 'gosales transactions for naive bayes model', 'healthcare python streaming application demo', 'use r dataframes & ibm watson natural language understanding', 'use xgboost, scikit-learn & ibm watson machine learning apis'])
assert get_ranked_article_unique_counts([1320, 232, 844])[0] == [1320, 123], "Oops! Your the get_ranked_article_unique_counts function doesn't work quite how we expect.\nMake sure you are using the user_item matrix to create the article counts."
print("If this is all you see, you passed all of our tests!  Nice job!")

If this is all you see, you passed all of our tests!  Nice job!


`4.` Now we are going to improve the consistency of the **user_user_recs** function from above.  

* Instead of arbitrarily choosing when we obtain users who are all the same closeness to a given user - choose the users that have the most total article interactions before choosing those with fewer article interactions.


* Instead of arbitrarily choosing articles from the user where the number of recommended articles starts below m and ends exceeding m, choose articles with the articles with the most total interactions before choosing those with fewer total interactions. This ranking should be  what would be obtained from the **top_articles** function you wrote earlier.

In [29]:
def get_top_sorted_users(user_id, user_item=user_item):
    """
    INPUT:
    user_id - (int)
    user_item - (pandas dataframe) matrix of users by articles:
            1's when a user has interacted with an article, 0 otherwise


    OUTPUT:
    neighbors_df - (pandas dataframe) a dataframe with:
                    neighbor_id - is a neighbor user_id
                    similarity - measure of the similarity of each user to the provided user_id
                    num_interactions - the number of articles viewed by the user

    Other Details - sort the neighbors_df by the similarity and then by number of interactions where
                    highest of each is higher in the dataframe, i.e. Descending order
                    Rows tied on both are listed by neighbor_id, smallest first.

    """
    # Hint: find similar users, but include their similarity, from there we can find the number of interactions for each neighbor
    pairs = find_similar_users(user_id, user_item=user_item, include_similarity=True)
    neighbors_df = pd.DataFrame(pairs, columns=["neighbor_id", "similarity"])
    articles_per_user = user_item.sum(axis=1)
    neighbors_df["num_interactions"] = neighbors_df["neighbor_id"].map(articles_per_user)
    neighbors_df = neighbors_df.sort_values(
        ["similarity", "num_interactions", "neighbor_id"],
        ascending=[False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    return neighbors_df # Return the dataframe specified in the doc_string


def user_user_recs_part2(user_id, m=10):
    """
    INPUT:
    user_id - (int) a user id
    m - (int) the number of recommendations you want for the user

    OUTPUT:
    recs - (list) a list of recommendations for the user by article id
    rec_names - (list) a list of recommendations for the user by article title

    Description:
    Loops through the users based on closeness to the input user_id
    For each user - finds articles the user hasn't seen before and provides them as recs
    Does this until m recommendations are found

    Notes:
    * Choose the users that have the most total article interactions
    before choosing those with fewer article interactions.

    * Choose articles with the articles with the most total interactions
    before choosing those with fewer total interactions.

    """
    # Article order from the Part II ranking: most interactions first.
    article_rank = {
        int(a): position
        for position, a in enumerate(_article_view_ranking(df)["article_id"])
    }
    seen = set(get_user_articles(user_id)[0])
    recs = []
    for neighbor in get_top_sorted_users(user_id)["neighbor_id"]:
        new = [a for a in get_user_articles(int(neighbor))[0] if a not in seen and a not in recs]
        recs.extend(sorted(new, key=article_rank.__getitem__))
        if len(recs) >= m:
            break
    recs = recs[:m]

    return recs, get_article_names(recs) # return your recommendations for this user_id

In [30]:
# Quick spot check - don't change this code - just use it to test your functions
rec_ids, rec_names = user_user_recs_part2(20, 10)
print("The top 10 recommendations for user 20 are the following article ids:")
print(rec_ids)
print()
print("The top 10 recommendations for user 20 are the following article names:")
print(rec_names)

The top 10 recommendations for user 20 are the following article ids:
[1293, 1162, 1185, 1165, 254, 1271, 1328, 1410, 40, 1402]

The top 10 recommendations for user 20 are the following article names:
['finding optimal locations of new store using decision optimization', 'analyze energy consumption in buildings', 'classify tumors with machine learning', 'analyze precipitation data', 'apple, ibm add machine learning to partnership with watson-core ml coupling', 'customer demographics and sales', 'income (2015): united states demographic measures', 'uci: sms spam collection', 'ensemble learning to improve machine learning results', 'uci: adult - predict income']


`5.` Use your functions from above to correctly fill in the solutions to the dictionary below.  Then test your dictionary against the solution.  Provide the code you need to answer each following the comments below.

In [31]:
print(get_top_sorted_users(1, user_item=user_item).head(n=1))
print(get_top_sorted_users(2, user_item=user_item).head(n=10))
print(get_top_sorted_users(131, user_item=user_item).head(n=10))

   neighbor_id  similarity  num_interactions
0         3933    0.986013                35
   neighbor_id  similarity  num_interactions
0         5083    0.730297                 5
1         1552    0.577350                 2
2         1890    0.577350                 2
3         1372    0.471405                 3
4         2941    0.433013                 8
5         3586    0.408248                 4
6          331    0.408248                 1
7          348    0.408248                 1
8          378    0.408248                 1
9          496    0.408248                 1
   neighbor_id  similarity  num_interactions
0         3870    0.986667                75
1          203    0.388909                96
2         4459    0.388909                96
3         3782    0.387585               135
4           40    0.384308                52
5         4932    0.384308                52
6           23    0.377647               135
7          242    0.375823                59
8         

In [32]:
### Tests with a dictionary of results
user1_most_sim = int(get_top_sorted_users(1).loc[0, "neighbor_id"])  # Find the user that is most similar to user 1
user2_6th_sim = int(get_top_sorted_users(2).loc[5, "neighbor_id"])  # Find the 6th most similar user to user 2
user131_10th_sim = int(get_top_sorted_users(131).loc[9, "neighbor_id"])  # Find the 10th most similar user to user 131

In [33]:
## Dictionary Test Here
sol_5_dict = {
    'The user that is most similar to user 1.': user1_most_sim, 
    'The user that is the 6th most similar to user 2.': user2_6th_sim,
    'The user that is the 10th most similar to user 131.': user131_10th_sim,
}

t.sol_5_test(sol_5_dict)

This all looks good!  Nice job!


**An exact tie at one of these positions.** For user 2, several neighbours have exactly the same cosine similarity: its square is 1/6. The cell below lists every one of them with its place in the ranking. They are told apart only in the computer's last binary digit, because the same value reached from different whole numbers rounds slightly differently (2 divided by the square root of 24 is stored one step higher than 3 divided by the square root of 54). The course's answer for the 6th neighbour follows that rounding. By the rule in question 4 (more interactions first), the tied user with the most articles would come first instead. The function computes the similarity in a way that rounds identically on every machine, so the answer below does not change between computers.

In [34]:
from fractions import Fraction

ranked = get_top_sorted_users(2)
user2_row = user_item.loc[2].to_numpy()
shared = user_item.loc[ranked["neighbor_id"]].to_numpy() @ user2_row
exact = [Fraction(int(k) * int(k), int(user2_row.sum()) * int(n)) for k, n in zip(shared, ranked["num_interactions"])]
ranked["position"] = ranked.index + 1
ranked["exact_cos_squared"] = [str(f) for f in exact]
ranked["stored_similarity"] = ranked["similarity"].map(repr)
tied = ranked[[f == Fraction(1, 6) for f in exact]]
print(f"Neighbours of user 2 whose cosine squared is exactly 1/6: {len(tied)}")
print(tied[["position", "neighbor_id", "num_interactions", "exact_cos_squared", "stored_similarity"]].head(8).to_string(index=False))
most = tied.sort_values("num_interactions", ascending=False, kind="mergesort").iloc[0]
print(f"Tied neighbour with the most articles: {int(most.neighbor_id)} ({int(most.num_interactions)} articles), ranked at position {int(most.position)}")

Neighbours of user 2 whose cosine squared is exactly 1/6: 38
 position  neighbor_id  num_interactions exact_cos_squared  stored_similarity
        6         3586                 4               1/6 0.4082482904638631
        7          331                 1               1/6 0.4082482904638631
        8          348                 1               1/6 0.4082482904638631
        9          378                 1               1/6 0.4082482904638631
       10          496                 1               1/6 0.4082482904638631
       11          504                 1               1/6 0.4082482904638631
       12          701                 1               1/6 0.4082482904638631
       13          861                 1               1/6 0.4082482904638631
Tied neighbour with the most articles: 4230 (9 articles), ranked at position 43


`6.` If we were given a new user, which of the above functions would you be able to use to make recommendations?  Explain.  Can you think of a better way we might make recommendations?  Use the cell below to explain a better method for new users.

**New users.** None of the user-user functions can help a brand-new user. `find_similar_users`, `get_top_sorted_users`, `user_user_recs` and `user_user_recs_part2` all start from the user's own row in `user_item`, and a new user has no row (or a row of zeros, whose similarity to everyone is undefined). The only functions that work without any history are the rank-based ones from Part II, `get_top_articles` and `get_top_article_ids`, so a new user is shown the most-viewed articles.

**A better way for new users.** Popularity is a reasonable first screen, but it can be improved without needing history:

* Count distinct readers rather than raw views, so a few people re-opening one article cannot push it up (Part II shows how different the two orders are).
* Weight recent interactions more heavily, so the list follows what people read now rather than what was popular when the platform started.
* Ask one or two questions at sign-up (a topic such as deep learning, Spark or data visualisation) and show the most-read articles within that topic, using the title clusters from Part IV.
* As soon as the user opens one article, switch to the content-based recommendations of Part IV, which need only that one article.

`7.` Using your existing functions, provide the top 10 recommended articles you would provide for the a new user below.  You can test your function against our thoughts to make sure we are all on the same page with how we might make a recommendation.

In [35]:
# What would your recommendations be for this new user 0?  As a new user, they have no observed articles.
# Provide a list of the top 10 article ids you would give to
new_user_recs = get_top_article_ids(10) # Your recommendations here



In [36]:
assert set(new_user_recs) == {1314, 1429, 1293, 1427, 1162, 1364, 1304, 1170, 1431, 1330}, "Oops!  It makes sense that in this case we would want to recommend the most popular articles, because we don't know anything about these users."

print("That's right!  Nice job!")

That's right!  Nice job!


### <a class="anchor" id="Content-Recs">Part IV: Content Based Recommendations</a>

Another method we might use to make recommendations is to recommend similar articles that are possibly related. One way we can find article relationships is by clustering text about those articles.  Let's consider content to be the article **title**, as it is the only text we have available. One point to highlight, there isn't one way to create a content based recommendation, especially considering that text information can be processed in many ways.  

`1.` Use the function bodies below to create a content based recommender function `make_content_recs`. We'll use TF-IDF to create a matrix based off article titles, and use this matrix to create clusters of related articles. You can use this function to make recommendations of new articles.

In [37]:
df.head()

,Unnamed: 0,article_id,title,user_id
0,0,1430,"using pixiedust for fast, flexible, and easier...",1
1,1,1314,healthcare python streaming application demo,2
2,2,1429,use deep learning for image classification,3
3,3,1338,ml optimization using cognitive assistant,4
4,4,1276,deploy your python model as a restful api,5


In [38]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD

In [39]:
# unique articles
df_unique_articles = (
    df.drop_duplicates("article_id")[["article_id", "title"]]
    .sort_values("article_id")
    .reset_index(drop=True)
)

In [40]:
# Create a vectorizer using TfidfVectorizer and fit it to the article titles
max_features = 200
max_df = 0.75
min_df = 5

vectorizer = TfidfVectorizer(
    max_df=max_df,
    min_df=min_df,
    stop_words="english",
    max_features=max_features,
)
print("Running TF-IDF")
X_tfidf = vectorizer.fit_transform(df_unique_articles["title"]) # Fit the vectorizer to the article titles

print(f"n_samples: {X_tfidf.shape[0]}, n_features: {X_tfidf.shape[1]}")

lsa = make_pipeline(TruncatedSVD(n_components=50, random_state=42), Normalizer(copy=False))
X_lsa = lsa.fit_transform(X_tfidf) # Fit the LSA model to the vectorized article titles
explained_variance = lsa[0].explained_variance_ratio_.sum()

print(f"Explained variance of the SVD step: {explained_variance * 100:.1f}%")

Running TF-IDF
n_samples: 714, n_features: 125
Explained variance of the SVD step: 76.0%


**Titles that leave no words behind.** `min_df=5` keeps only words that appear in at least five titles, and stop words are removed. Some titles have no word left, so their TF-IDF row is all zeros and they carry no text for the clusters to use. The cell below counts them and shows a few.

In [41]:
empty_rows = np.asarray(X_tfidf.sum(axis=1)).ravel() == 0
print(f"Titles with no remaining words: {int(empty_rows.sum())} of {X_tfidf.shape[0]}")
print(df_unique_articles.loc[empty_rows, "title"].head(5).to_string())

Titles with no remaining words: 49 of 714
31                                  readr 1.0.0
34                         interconnect with us
41             transform anything into a vector
66    feature importance and why it's important
77             how to solve 90% of nlp problems


In [42]:
# Let's map the inertia for different number of clusters to find the optimal number of clusters
# We'll plot it to see the elbow
inertia = []
clusters = 300
step = 25
max_iter = 50
n_init = 5
random_state = 42
for k in range(1, clusters, step):
    kmeans = KMeans(
        n_clusters=k,
        max_iter=max_iter,
        n_init=n_init,
        random_state=random_state,
    ).fit(X_lsa)
    # inertia is the sum of squared distances to the closest cluster center
    inertia.append(kmeans.inertia_)
plt.plot(range(1, clusters, step), inertia)
plt.xlabel('Number of clusters')

Text(0.5, 0, 'Number of clusters')

There appears to be an elbow about 50, so we'll use 50 clusters.

**Checking the elbow.** The inertia falls steadily rather than bending sharply at one point, so the elbow has to be located rather than seen. The cell below finds the sampled number of clusters that lies furthest from the straight line joining the first and last points of the curve (both axes scaled to 0-1), which is where the curve bends most. The grid steps by 25, so the answer can only be one of the sampled values, and the code below uses whatever it finds rather than a typed-in number.

In [43]:
def elbow_point(ks, values):
    """Return the k whose point lies furthest below the chord of the curve.

    INPUT:
    ks - (sequence of int) the sampled numbers of clusters, ascending
    values - (sequence of float) the inertia at each k

    OUTPUT:
    k - (int) the sampled k with the largest distance from the straight line
        joining the first and last points, after scaling both axes to 0-1
    """
    x = np.asarray(ks, dtype=float)
    y = np.asarray(values, dtype=float)
    x = (x - x.min()) / (x.max() - x.min())
    y = (y - y.min()) / (y.max() - y.min())
    distance = (1 - x) - y  # the chord runs from (0, 1) to (1, 0)
    return int(np.asarray(ks)[np.argmax(distance)])


sampled_ks = list(range(1, clusters, step))
elbow_k = elbow_point(sampled_ks, inertia)
print(f"Furthest point from the chord: k = {elbow_k}")

Furthest point from the chord: k = 51


In [44]:
n_clusters = elbow_k # Number of clusters
kmeans = KMeans(
    n_clusters=n_clusters,
    max_iter=max_iter,
    n_init=n_init,
    random_state=random_state,
).fit(X_lsa)

**Are these clusters stable?** k-means starts from random centres, so a different seed can give different clusters. Cluster numbers are arbitrary between fits, so the labels are compared with the adjusted Rand index, which ignores the numbering (1 means the same grouping, about 0 means no better than chance). The cell below refits with three other seeds and compares each with the clusters used here. It also shows where the titles with no remaining words ended up.

In [45]:
from sklearn.metrics import adjusted_rand_score

for seed in (0, 1, 7):
    other = KMeans(n_clusters=n_clusters, max_iter=max_iter, n_init=n_init, random_state=seed).fit(X_lsa)
    print(f"seed {seed}: adjusted Rand index {adjusted_rand_score(kmeans.labels_, other.labels_):.3f}")

empty_clusters = sorted(set(kmeans.labels_[empty_rows].tolist()))
print(f"Clusters holding the titles with no remaining words: {empty_clusters}")
for c in empty_clusters:
    members = int((kmeans.labels_ == c).sum())
    print(f"cluster {c}: {members} articles, {int((kmeans.labels_[empty_rows] == c).sum())} of them with no remaining words")

seed 0: adjusted Rand index 0.826
seed 1: adjusted Rand index 0.797
seed 7: adjusted Rand index 0.768
Clusters holding the titles with no remaining words: [4]
cluster 4: 62 articles, 49 of them with no remaining words


In [46]:
# create a new column `title_cluster` and assign it the kmeans cluster labels
# First we need to map the labels to df_unique_articles article ids and then apply those to df
article_cluster_map = dict(zip(df_unique_articles["article_id"], kmeans.labels_.tolist())) # Map article_id to cluster label
df['title_cluster'] = df["article_id"].map(article_cluster_map) # apply map to create title clusters

In [47]:
# Let's check the number of articles in each cluster
np.array(np.unique(kmeans.labels_, return_counts=True)).T

array([[ 0, 10],
       [ 1,  4],
       [ 2, 10],
       [ 3, 12],
       [ 4, 62],
       [ 5,  8],
       [ 6, 20],
       [ 7, 35],
       [ 8, 28],
       [ 9, 19],
       [10, 16],
       [11, 13],
       [12, 42],
       [13, 14],
       [14,  9],
       [15, 23],
       [16, 27],
       [17, 22],
       [18, 19],
       [19, 20],
       [20, 15],
       [21,  9],
       [22,  7],
       [23, 10],
       [24,  8],
       [25, 22],
       [26, 11],
       [27,  8],
       [28, 11],
       [29,  8],
       [30,  8],
       [31,  7],
       [32,  9],
       [33,  7],
       [34,  8],
       [35,  6],
       [36, 10],
       [37,  5],
       [38,  6],
       [39, 10],
       [40,  8],
       [41, 17],
       [42,  6],
       [43,  9],
       [44, 13],
       [45,  9],
       [46, 11],
       [47, 15],
       [48,  7],
       [49, 11],
       [50, 10]])

In [48]:
def get_similar_articles(article_id, df=df):
    """
    INPUT:
    article_id - (int) an article id
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    article_ids - (list) a list of article ids that are in the same title cluster

    Description:
    Returns a list of the article ids that are in the same title cluster
    """
    title_cluster = article_cluster_map[int(article_id)]
    articles_in_cluster = sorted(
        int(a) for a, c in article_cluster_map.items() if c == title_cluster
    )

    # remove the input article_id from the list
    articles_in_cluster = [a for a in articles_in_cluster if a != int(article_id)]

    return articles_in_cluster

In [49]:
def make_content_recs(article_id, n, df=df):
    """
    INPUT:
    article_id - (int) an article id
    n - (int) the number of recommendations you want similar to the article id
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    n_ranked_similar_articles - (list) a list of article ids that are in the same title cluster ranked
                                by popularity
    n_ranked_article_names - (list) a list of article names associated with the list of article ids

    Description:
    Returns a list of the n most ranked similar articles to a given article_id based on the title
    cluster in df. Rank similar articles using the function get_ranked_article_unique_counts.
    """
    similar = get_similar_articles(article_id, df=df)
    ranked = get_ranked_article_unique_counts(similar)
    n_ranked_similar_articles = [a for a, _ in ranked[:n]]
    n_ranked_article_names = get_article_names(n_ranked_similar_articles, df=df)

    return n_ranked_similar_articles, n_ranked_article_names
        

In [50]:
# Test out your content recommendations given artice_id 25
rec_article_ids, rec_article_titles = make_content_recs(25, 10)
print(rec_article_ids)
print(rec_article_titles)

[1025, 101, 975, 766, 508, 547, 132, 878, 92, 693]
['data tidying in data science experience', 'how to choose a project to practice data science', 'the data science process', 'making data science a team sport', 'data science in the cloud', 'trust in data science', 'collecting data science cheat sheets', '10 data science podcasts you need to be listening to right now', '9 mistakes to avoid when starting your career in data science', 'better together: spss and data science experience']


In [51]:
assert len({1025, 593, 349, 821, 464, 29, 1042, 693, 524, 352}.intersection(set(rec_article_ids))) > 0, "Oops! Your the make_content_recs function doesn't work quite how we expect."

`2.` Now that you have put together your content-based recommendation system, use the cell below to write a summary explaining how your content based recommender works.  Do you see any possible improvements that could be made to your function? What other text data would be useful to help make better recommendations besides the article title?

**How the content-based recommender works.**

1. Each article is described by its title only, the only text in this data.
2. The titles are turned into TF-IDF vectors: a word scores highly for a title when it appears there and is rare across titles. English stop words are dropped, and only words found in at least five titles (and at most 75% of them) are kept.
3. Latent semantic analysis (a truncated SVD to 50 dimensions, then scaling each vector to length 1) groups words that tend to appear together, so titles about the same topic land close together even when they use different words.
4. k-means groups the 714 articles into clusters of similar titles. The number of clusters was read from the inertia curve as the point where it bends most.
5. `make_content_recs(article_id, n)` finds the article's cluster, removes the article itself, and returns the `n` cluster members read by the most distinct users.

**What could be better.**

* Some titles keep no words at all after the vocabulary filter (counted above). Their vectors are identical, so they all fall into the same cluster (shown above), and for those articles the recommendations are decided by popularity, not content. Lowering `min_df`, or treating those articles separately, would fix it.
* The clusters change with the random seed (the adjusted Rand index between seeds above is clearly below 1), so a run with a different seed would recommend partly different articles.
* A cluster is all-or-nothing: an article on the edge of a cluster is treated the same as one at its centre. Ranking cluster members by cosine similarity to the article in the LSA space, then by popularity, would put the closest articles first.
* Recommendations never cross a cluster boundary, even when the nearest article sits just outside it.

**Other text that would help.** The article body, or at least its first paragraph; the abstract or description shown on the platform; tags or categories; the author; the programming language or IBM product the article uses; and the publication date, so that newer articles on the same topic can be preferred.

### Extra: a stronger content-based recommender

The cluster recommender above has three weaknesses, all measured in Part IV: titles with no words left after the vocabulary filter, clusters that change with the random seed, and all-or-nothing cluster membership. This version changes two things:

* **Text that no title loses.** Besides whole words (and pairs of words) found in at least two titles, each title is also described by its character sequences of length 3 to 5. "readr 1.0.0" or "pixiedust" share no whole word with other titles, but they share pieces of words. Both kinds of feature are scaled down for very common terms (TF-IDF), reduced to 200 latent dimensions (LSA) and scaled to length 1.
* **Nearest neighbours instead of clusters.** Each article's recommendations are the articles whose vectors point most nearly the same way (highest cosine similarity), so there is no cluster boundary to fall on the wrong side of, and no random seed in the recommendation step.

In [52]:
import scipy.sparse as sparse

word_tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
X_text = sparse.hstack([
    word_tfidf.fit_transform(df_unique_articles["title"]),
    char_tfidf.fit_transform(df_unique_articles["title"]),
]).tocsr()
text_lsa = make_pipeline(TruncatedSVD(n_components=200, random_state=42), Normalizer(copy=False))
X_text_lsa = text_lsa.fit_transform(X_text)
title_similarity = X_text_lsa @ X_text_lsa.T
article_position = {int(a): i for i, a in enumerate(df_unique_articles["article_id"])}
print(f"Features: {X_text.shape[1]} (words and word pairs: {len(word_tfidf.vocabulary_)}, character pieces: {len(char_tfidf.vocabulary_)})")
print(f"Titles with no features left: {int((np.asarray(X_text.sum(axis=1)).ravel() == 0).sum())} of {X_text.shape[0]}")


def make_content_recs_nn(article_id, n, df=df):
    """
    INPUT:
    article_id - (int) an article id
    n - (int) the number of recommendations you want similar to the article id
    df - (pandas dataframe) df as defined at the top of the notebook

    OUTPUT:
    ids - (list) the n article ids whose titles are most similar to this
          article's title, most similar first (ties by article id)
    names - (list) their titles

    Description:
    Ranks every other article by the cosine similarity of its title vector
    (words, word pairs and character pieces, reduced by LSA) to the given
    article's, and returns the top n.
    """
    row = title_similarity[article_position[int(article_id)]]
    order = sorted(
        (i for i in range(len(row)) if i != article_position[int(article_id)]),
        key=lambda i: (-row[i], int(df_unique_articles["article_id"].iloc[i])),
    )
    ids = [int(df_unique_articles["article_id"].iloc[i]) for i in order[:n]]
    return ids, get_article_names(ids, df=df)


for example in (25, int(df_unique_articles.loc[empty_rows, "article_id"].iloc[0])):
    ids, names = make_content_recs_nn(example, 5)
    print(f"\n{get_article_names([example])[0]!r} ->")
    for a, name in zip(ids, names):
        print(f"   {a}: {name}")

Features: 7385 (words and word pairs: 711, character pieces: 6674)
Titles with no features left: 0 of 714

'creating the data science experience' ->
   48: data science experience documentation
   1025: data tidying in data science experience
   644: h2o with ibm's data science experience (dsx)
   416: using the maker palette in the ibm data science experience
   693: better together: spss and data science experience

'readr 1.0.0' ->
   256: xml2 1.0.0
   314: some random weekend reading
   400: read and write data to and from amazon s3 buckets in rstudio
   617: pixiedust gets its first community-driven feature in 1.0.4
   1042: use ibm data science experience to read and write data stored on amazon s3


**Do the recommendations match what people read?** Titles cannot show whether a recommendation is useful, but reading behaviour gives a rough check: articles that are truly related should tend to be read by the same people. For every article, the cell below takes its ten recommendations and measures the overlap of their readers with the article's readers (the Jaccard index: readers of both, divided by readers of either). It compares four methods: the nearest-neighbour recommender, the cluster recommender, the ten most-read articles, and ten articles drawn at random. It also counts how many different articles each method ever recommends.

This is a proxy, and it favours popular articles, whose readers overlap with everyone's; that is why popularity is in the table.

In [53]:
readers_matrix = user_item[[int(a) for a in df_unique_articles["article_id"]]].to_numpy(dtype=bool)
reader_counts = readers_matrix.sum(axis=0)
most_read = sorted(range(len(reader_counts)), key=lambda i: (-reader_counts[i], i))
random_picks = np.random.default_rng(42)


def reader_overlap(recommend):
    """Mean reader Jaccard index of each article with its recommendations.

    INPUT:
    recommend - (function) position of an article -> positions of its ten
                recommendations in df_unique_articles

    OUTPUT:
    per_article - (numpy array) mean Jaccard index per article (NaN when a
                  method returns nothing for it)
    covered - (int) how many different articles the method recommends
    """
    per_article, covered = [], set()
    for i in range(len(reader_counts)):
        picks = list(recommend(i))
        covered.update(picks)
        if not picks:
            per_article.append(np.nan)
            continue
        both = (readers_matrix[:, [i]] & readers_matrix[:, picks]).sum(axis=0)
        either = (readers_matrix[:, [i]] | readers_matrix[:, picks]).sum(axis=0)
        per_article.append(float(np.mean(both / either)))
    return np.array(per_article), len(covered)


methods = {
    "nearest neighbours": lambda i: [article_position[a] for a in make_content_recs_nn(int(df_unique_articles["article_id"].iloc[i]), 10)[0]],
    "clusters (Part IV)": lambda i: [article_position[a] for a in make_content_recs(int(df_unique_articles["article_id"].iloc[i]), 10)[0]],
    "most read": lambda i: [j for j in most_read if j != i][:10],
    "random": lambda i: list(random_picks.choice([j for j in range(len(reader_counts)) if j != i], 10, replace=False)),
}
rows_out = []
for name, recommend in methods.items():
    overlap, covered = reader_overlap(recommend)
    rows_out.append([name, np.nanmean(overlap), np.nanmean(overlap[empty_rows]), covered])
comparison = pd.DataFrame(rows_out, columns=["method", "overlap, all articles", "overlap, no-word titles", "articles ever recommended"]).set_index("method")
print(comparison.round(4).to_string())

                    overlap, all articles  overlap, no-word titles  articles ever recommended
method                                                                                       
nearest neighbours                 0.0232                   0.0155                        714
clusters (Part IV)                 0.0234                   0.0134                        480
most read                          0.0207                   0.0152                         11
random                             0.0089                   0.0060                        714


**What the comparison shows.** On reader overlap across all articles the two content methods are level (0.0232 for nearest neighbours, 0.0234 for clusters). Both are above the most-read list (0.0207) and well above random picks (0.0089), so title similarity does follow what people read together. The gains are elsewhere:

* Every title now has features (none left empty, against 49 before). On those 49 articles, the nearest neighbours' readers overlap more with the article's own (0.0155) than the cluster method's picks do (0.0134).
* All 714 articles can be recommended, against 480 for the clusters and 11 for the most-read list, so less-read articles get shown at all.
* The recommendations no longer depend on a k-means seed.

The examples above show the character pieces at work: "readr 1.0.0" is matched with "xml2 1.0.0", another R package release, although the two titles share no whole word. Matching on pieces of words has a cost too: "some random weekend reading" is among its five, most likely through the letters "read". Article bodies or tags, if they were available, would give the method more to go on than titles.

### <a class="anchor" id="Matrix-Fact">Part V: Matrix Factorization</a>

In this part of the notebook, you will build use matrix factorization to make article recommendations to users.

`1.` You should have already created a **user_item** matrix above in **question 1** of **Part III** above.  This first question here will just require that you run the cells to get things set up for the rest of **Part V** of the notebook. 

In [54]:
# quick look at the matrix
user_item.head()

article_id,0,2,4,8,9,12,14,15,16,18,...,1434,1435,1436,1437,1439,1440,1441,1442,1443,1444
user_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


`2.` In this situation, you can use Singular Value Decomposition from [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) on the user-item matrix.  Use the cell to perform SVD.

In [55]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import precision_score, recall_score, accuracy_score
# Using the full number of components which equals the number of columns
svd = TruncatedSVD(n_components=len(user_item.columns), n_iter=5, random_state=42)

u = svd.fit_transform(user_item)
v = svd.components_
s = svd.singular_values_ 
print('u', u.shape)
print('s', s.shape)
print('vt', v.shape)

u (5149, 714)
s (714,)
vt (714, 714)


**Why SVD works in this case.** In the lessons, plain SVD could not be used on a ratings matrix, because most ratings are missing and SVD needs every cell filled in; FunkSVD was used instead, fitting only the known ratings. This matrix has no missing cells: every user-article pair is either 1 (the user opened the article) or 0 (they did not), so SVD can factorise it directly. It splits the matrix into U (users by latent features), Sigma (how strong each latent feature is) and V-transpose (latent features by articles). Each latent feature is a group of articles that tend to be read by the same people, so keeping only the strongest features keeps the shared reading patterns and drops the detail of single users. The cost of treating every gap as 0 is that "not opened yet" is handled the same as "not interested", which is why the scores below need care.

**What `u` holds.** `TruncatedSVD.fit_transform` returns U already multiplied by Sigma, which is why the metrics cell below rebuilds the matrix with `np.dot(u_new, vt_new)` and no separate Sigma. The cell below separates the three matrices and checks them. Not all 714 components carry information: some articles were read by exactly the same users, so their columns are identical, and the matrix has fewer independent columns than articles. The cell counts the singular values that are zero; U is only defined (and orthonormal) for the others. U times Sigma times V-transpose still rebuilds the 0/1 matrix to within rounding error.

In [56]:
nonzero = s > 1e-10 * s.max()
print(f"Matrix rank: {np.linalg.matrix_rank(user_item.values.astype(float))}")
print(f"Singular values above zero: {int(nonzero.sum())} of {len(s)}")
print(f"Articles whose column duplicates another article's: {int(user_item.T.duplicated().sum())}")
U = np.zeros_like(u)
U[:, nonzero] = u[:, nonzero] / s[nonzero]   # undo the Sigma that fit_transform applied
Sigma = np.diag(s)
Vt = v
print('U', U.shape, 'Sigma', Sigma.shape, 'Vt', Vt.shape)
kept = U[:, nonzero]
print(f"Largest departure of U's columns from orthonormal (non-zero singular values): {np.max(np.abs(kept.T @ kept - np.eye(kept.shape[1]))):.2e}")
print(f"Largest rebuild error of U Sigma Vt, all components: {np.max(np.abs(U @ Sigma @ Vt - user_item.values)):.2e}")

Matrix rank: 704
Singular values above zero: 704 of 714
Articles whose column duplicates another article's: 8
U (5149, 714) Sigma (714, 714) Vt (714, 714)
Largest departure of U's columns from orthonormal (non-zero singular values): 2.58e-14
Largest rebuild error of U Sigma Vt, all components: 9.21e-15


`3.` Now for the tricky part, how do we choose the number of latent features to use?  Running the below cell, you can see that as the number of latent features increases, we obtain better metrics when making predictions for the 1 and 0 values in the user-item matrix.  Run the cell below to get an idea of how our metrics improve as we increase the number of latent features.

In [57]:
num_latent_feats = np.arange(10, 700+10, 20)
metric_scores = []

for k in num_latent_feats:
    # restructure with k latent features
    u_new, vt_new = u[:, :k], v[:k, :]
    
    # take dot product
    user_item_est = abs(np.around(np.dot(u_new, vt_new))).astype(int)
    # make sure the values are between 0 and 1
    user_item_est = np.clip(user_item_est, 0, 1)
    
    # total errors and keep track of them
    acc = accuracy_score(user_item.values.flatten(), user_item_est.flatten())
    precision = precision_score(user_item.values.flatten(), user_item_est.flatten())
    recall = recall_score(user_item.values.flatten(), user_item_est.flatten())
    metric_scores.append([acc, precision, recall])
    
    
plt.plot(num_latent_feats, metric_scores, label=['Accuracy', 'Precision', 'Recall'])
plt.legend()
plt.xlabel('Number of Latent Features')
plt.title('Metrics vs. Number of Latent Features')

Text(0.5, 1.0, 'Metrics vs. Number of Latent Features')

**What these scores are measured on.** The metrics above score the model on the same matrix it was fitted to, so more latent features can only fit it better. Only about 1% of the matrix is 1s, so accuracy starts near its ceiling: predicting no interaction anywhere already scores the accuracy printed below. Precision and recall, which only look at the 1s, are the useful lines on this plot. A 0 in this matrix means the user has not opened the article, not that they disliked it: the data records interactions, never ratings, so a 0 is unknown rather than negative. The cell below prints that floor and the scores at a few numbers of latent features.

In [58]:
all_zero_accuracy = 1 - user_item.values.mean()
print(f"Share of the matrix that is 1: {user_item.values.mean():.4f}")
print(f"Accuracy of predicting no interaction anywhere: {all_zero_accuracy:.4f}")
scores = pd.DataFrame(metric_scores, index=num_latent_feats, columns=["accuracy", "precision", "recall"])
print(scores.loc[[10, 50, 110, 150, 190, 210, 310, 410, 510, 610, 690]].round(4).to_string())

Share of the matrix that is 1: 0.0092
Accuracy of predicting no interaction anywhere: 0.9908
     accuracy  precision  recall
10     0.9917     0.8460  0.1142
50     0.9945     0.9887  0.4070
110    0.9969     0.9997  0.6600
150    0.9979     0.9999  0.7670
190    0.9985     1.0000  0.8321
210    0.9987     1.0000  0.8590
310    0.9995     1.0000  0.9427
410    0.9998     1.0000  0.9787
510    0.9999     1.0000  0.9945
610    1.0000     1.0000  0.9994
690    1.0000     1.0000  1.0000


`4.` From the above, we can't really be sure how many features to use, because simply having a better way to predict the 1's and 0's of the matrix doesn't exactly give us an indication of if we are able to make good recommendations. Given the plot above, what would you pick for the number of latent features and why?

**Choice: 200 latent features.**

Reading the plot and the table above it:

* **Accuracy** is close to 1 everywhere (0.9917 at 10 features), and predicting no interaction at all already scores 0.9908, so it says almost nothing about the choice.
* **Precision** reaches 0.9887 by 50 features and about 1 from 110 on: once rounded to 0 or 1, almost every 1 the model predicts really is a 1 in the matrix it was fitted to.
* **Recall** is the line that moves. It is 0.4070 at 50 features, 0.6600 at 110, 0.8321 at 190 and 0.8590 at 210, and each extra 100 features adds less than the last. The first features capture the popular articles that many users share; later ones recover the histories of smaller and smaller groups, until all the features rebuild the matrix exactly (recall 1.0000 at 690), which is memorising it rather than learning anything that generalises.

200 sits between the sampled 190 and 210 on the flattening part of the recall curve: it recovers more than four in five of the observed interactions while using under a third of the 714 possible features, and it is the number question 5 asks for.

The limit of this choice is that every score on the plot is measured on the data the model was fitted to, so more features always look better. Choosing properly needs interactions the model has not seen: hold some out, fit on the rest, and pick the number of features where held-out precision and recall stop improving. The extra section at the end of this notebook does that with the course's own train/test split, and checks this choice against it.

`5.` Using 200 latent features and the values of U, S, and V transpose we calculated above, create an article id recommendation function that finds similar article ids to the one provide.

Create a list of 10 recommendations that are similar to article with id 4.  The function should provide these recommendations by finding articles that have the most similar latent features as the provided article.

In [59]:
def get_svd_similar_article_ids(article_id, vt, user_item=user_item, include_similarity=False):
    """
    INPUT:
    article_id - (int) an article id
    vt - (numpy array) vt matrix from SVD
    user_item - (pandas dataframe) matrix of users by articles:
                1's when a user has interacted with an article, 0 otherwise
    include_similarity - (bool) whether to include the similarity in the output

    OUTPUT:
    article_ids - (list) a list of article ids ordered by how similar their
                  latent features are to the given article's, most similar
                  first, without the article itself

    Description:
    Returns a list of the article ids similar using SVD factorization
    Articles whose latent-feature column is only rounding noise are treated
    as having no direction (similarity 0), so the order is the same on
    every machine.
    """
    # Find the index of the article_id
    article_idx = list(user_item.columns).index(article_id)
    # An article that none of the kept latent features describe has a column
    # of rounding noise (about 1e-17 long). Its direction means nothing and
    # differs from one computer to the next, so it is set to zero: such an
    # article gets similarity 0 to every other article, on every machine.
    vt = np.asarray(vt, dtype=float)
    lengths = np.linalg.norm(vt, axis=0)
    vt = np.where(lengths < 1e-10 * lengths.max(), 0.0, vt)
    # Find the cosine similarity of all articles
    # Hint: vt should be transposed before passing to cosine_similarity to get a 714 x 714 matrix
    cos_sim = cosine_similarity(vt.T)
    # Get similarities only for the cos_sim of the article_idx
    similarities = pd.Series(cos_sim[article_idx], index=user_item.columns)

    # Sort and return the articles, don't include the own article
    similarities = similarities.drop(article_id).sort_values(ascending=False, kind="mergesort")
    most_similar_items = [int(a) for a in similarities.index]

    if include_similarity:
        return [[int(a), float(sim)] for a, sim in similarities.items()] # return a list of lists with [[similar_article_id, similarity], ...]
    return most_similar_items

In [60]:
# Create a vt_new matrix with 200 latent features
k = 200
vt_new = v[:k, :]

In [61]:
# What is the article name for article_id 4?
print("Current article:", get_article_names([4], df=df)[0])

Current article: analyze ny restaurant data using spark in dsx


In [62]:
# What are the top 10 most similar articles to article_id 4?
rec_articles = get_svd_similar_article_ids(4, vt_new, user_item=user_item)[:10]
rec_articles

[1199, 1068, 486, 1202, 176, 1120, 244, 793, 58, 132]

In [63]:
# What are the top 10 most similar articles to article_id 4?
get_article_names(rec_articles, df=df)

['country statistics: crude oil - exports',
 'airbnb data for analytics: athens reviews',
 'use spark r to load and analyze data',
 'country statistics: crude oil - proved reserves',
 'top analytics tools in 2016',
 'airbnb data for analytics: paris calendar',
 'notebooks: a power tool for data scientists',
 '10 powerful features on watson data platform, no coding necessary',
 'advancements in the spark community',
 'collecting data science cheat sheets']

In [64]:
assert set(rec_articles) == {1199, 1068, 486, 1202, 176, 1120, 244, 793, 58, 132}, "Oops! Your the get_svd_similar_article_ids function doesn't work quite how we expect."
print("That's right!  Great job!")

That's right!  Great job!


`6.` Use the cell below to comment on the results you found in the previous question. Given the circumstances of your results, discuss what you might do to determine if the recommendations you make above are an improvement to how users currently find articles, either by Sections 2, 3, or 4? Add any tradeoffs between each of the methods, and how you could leverage each type for different situations including new users with no history, recently new users with little history, and users with a lot of history. 

**What the SVD recommendations show.** Article 4 is about analysing New York restaurant data with Spark. Its ten closest articles in the 200-feature space are a mix of public-dataset articles (country statistics, Airbnb data) and Spark and analytics articles, and few of them share a word with its title. They are related through the people who read them, not through their words, so this method can find links that a title-based method cannot (two articles read by the same people but with no words in common), and it misses links that title clusters find (a brand-new article that nobody has read yet has no meaningful latent features).

**How I would test whether any of this helps users.** The measurements in this notebook are all offline, and the SVD ones are scored on the data the model was fitted to. Whether recommendations improve how people find articles can only be shown with users:

1. *Offline first.* Split interactions by time: fit on the earlier months, and measure precision and recall at 10 on what each user actually read later. Compare each method with the popularity baseline on the same split, and count the users each method cannot score at all. The extra section below does a first version of this with the course's split of the file.
2. *Then an A/B test.* Randomly assign users to the current experience or to a recommendation panel. The main metric is click-through on recommended articles; guard metrics are articles read per visit, return visits within a week, and the share of reads that come from outside the top-20 popular list (to check the panel is not just repeating Part II). Fix the sample size and duration in advance from a power calculation, and do not stop early because the numbers look good: checking repeatedly inflates false alarms.

**Trade-offs, and which method suits which user.**

| User | Best method | Why |
| --- | --- | --- |
| New, no history | Rank-based (Part II), ideally by distinct readers or within a chosen topic | Nothing else has any input to work with |
| New, one or a few articles read | Content-based (Part IV), then mix in popularity | Needs only one article; user-user and SVD have too little history to find real neighbours |
| Established, long history | User-user (Part III) or SVD (Part V), blended with content-based | Enough history to find similar users and latent tastes; content-based keeps new articles in the mix |

Rank-based is simple and robust but the same for everyone. Content-based handles new articles and new users but only sees titles here, and its clusters depend on the random seed. User-user collaborative filtering is easy to explain ("people who read what you read also read this") but needs history and compares every pair of users, which grows with the square of the user count. SVD compresses that comparison into a few hundred features and can relate articles with no words in common, but it cannot place new users or new articles until it is refitted.

### Extra: testing on interactions the model has not seen

Every score in question 3 was measured on the matrix the model was fitted to. This section holds some interactions out instead. As in the course's test file, the first 40,000 rows of the interaction table are the training set and the last 5,993 rows are the test set (the rows are split in the file's own order). A user or article the model never saw in training cannot be scored by SVD at all, which is the cold-start problem; the cells below count them.

In [65]:
def create_test_and_train_user_item(df_train, df_test):
    """Build the training and test user-item matrices from the two splits.

    INPUT:
    df_train - (pandas dataframe) the training rows of df
    df_test - (pandas dataframe) the test rows of df

    OUTPUT:
    user_item_train - (pandas dataframe) 0/1 user-item matrix of the training rows
    user_item_test - (pandas dataframe) 0/1 user-item matrix of the test rows
    test_idx - (list) the user ids in the test set
    test_arts - (list) the article ids in the test set
    """
    user_item_train = create_user_item_matrix(df_train)
    user_item_test = create_user_item_matrix(df_test)
    test_idx = [int(u) for u in user_item_test.index]
    test_arts = [int(a) for a in user_item_test.columns]
    return user_item_train, user_item_test, test_idx, test_arts


df_train = df.head(40000)
df_test = df.tail(5993)
user_item_train, user_item_test, test_idx, test_arts = create_test_and_train_user_item(df_train, df_test)

predictable_users = sorted(set(test_idx) & set(user_item_train.index))
predictable_arts = sorted(set(test_arts) & set(user_item_train.columns))
print(f"Test users: {len(test_idx)}, also in training: {len(predictable_users)}")
print(f"Test users the model cannot score (cold start): {len(test_idx) - len(predictable_users)}")
print(f"Test articles: {len(test_arts)}, also in training: {len(predictable_arts)}")

Test users: 682, also in training: 20
Test users the model cannot score (cold start): 662
Test articles: 574, also in training: 574


In [66]:
sol_4_dict = {
    'How many users can we make predictions for in the test set?': len(predictable_users),
    'How many users in the test set are we not able to make predictions for because of the cold start problem?': len(test_idx) - len(predictable_users),
    'How many articles can we make predictions for in the test set?': len(predictable_arts),
    'How many articles in the test set are we not able to make predictions for because of the cold start problem?': len(test_arts) - len(predictable_arts),
}

t.sol_4_test(sol_4_dict)

Awesome job!  That's right!  All of the test articles are in the training data, but there are only 20 test users that were also in the training set.  All of the other users that are in the test set we have no data on.  Therefore, we cannot make predictions for these users using SVD.


**Scoring the held-out interactions.** SVD is fitted on the training matrix only. For the users who appear in both sets, the model's scores for every test article are compared with what those users actually opened in the test rows. Two measures are shown:

* the course-style accuracy of the rounded 0/1 predictions, beside the accuracy of predicting no interaction at all;
* **precision at 10**: for each user, the ten highest-scoring articles they had not already opened in training, and the share of those ten they then opened in the test rows. Recall at 10 is the share of all the users' new test articles that made their ten. Re-reads of articles from training are left out, because recommending something already read is not a recommendation.

The same measures are given for the popularity ranking from Part II (counted on the training rows only), which every model has to beat.

In [67]:
svd_train = TruncatedSVD(n_components=min(user_item_train.shape) - 1, n_iter=5, random_state=42)
u_train = svd_train.fit_transform(user_item_train)
vt_train = svd_train.components_

rows = [list(user_item_train.index).index(user) for user in predictable_users]
cols = [list(user_item_train.columns).index(a) for a in predictable_arts]
held_out = user_item_test.loc[predictable_users, predictable_arts].values
seen = {user: set(user_item_train.columns[user_item_train.loc[user] == 1]) for user in predictable_users}
new_reads = {user: set(user_item_test.columns[user_item_test.loc[user] == 1]) - seen[user] for user in predictable_users}
train_views = df_train["article_id"].value_counts()


def hits_at_10(score_rows):
    """Per-user hits among the ten best-scoring articles not read in training.

    INPUT:
    score_rows - (2-d array) one row of scores per predictable user, one
                 column per predictable article

    OUTPUT:
    hits - (numpy array) for each user, how many of their ten picks they
           opened in the test rows. Ties are broken by article id.
    """
    hits = []
    for j, user in enumerate(predictable_users):
        order = sorted(range(len(predictable_arts)), key=lambda c: (-score_rows[j][c], predictable_arts[c]))
        picks = [predictable_arts[c] for c in order if predictable_arts[c] not in seen[user]][:10]
        hits.append(len(set(picks) & new_reads[user]))
    return np.array(hits)


total_new = sum(len(v) for v in new_reads.values())
popularity = np.array([[train_views.get(a, 0) for a in predictable_arts]] * len(predictable_users), dtype=float)
pop_hits = hits_at_10(popularity)
print(f"Held-out cells: {held_out.size}; ones: {int(held_out.sum())}; new articles read by the {len(predictable_users)} users: {total_new}")
print(f"Accuracy of predicting no interaction anywhere: {1 - held_out.mean():.4f}")
print(f"Popularity: precision@10 {pop_hits.mean() / 10:.3f}, recall@10 {pop_hits.sum() / total_new:.3f}")

held_out_scores = []
for k in num_latent_feats:
    est = u_train[rows, :k] @ vt_train[:k][:, cols]
    accuracy = (np.clip(np.abs(np.around(est)), 0, 1).astype(int) == held_out).mean()
    hits = hits_at_10(est)
    held_out_scores.append([k, accuracy, hits.mean() / 10, hits.sum() / total_new])
held_out_scores = pd.DataFrame(held_out_scores, columns=["k", "accuracy", "precision@10", "recall@10"]).set_index("k")
best_k = int(held_out_scores["precision@10"].idxmax())
print(held_out_scores.loc[[10, 50, 110, 150, 190, 210, 310, 410, 510, 690]].round(4).to_string())
print(f"Best held-out precision@10: k = {best_k} ({held_out_scores.loc[best_k, 'precision@10']:.3f})")
pop_precision = pop_hits.mean() / 10
better = [int(k) for k in held_out_scores.index if held_out_scores.loc[k, "precision@10"] > pop_precision]
gaps = [k for k in held_out_scores.index if min(better) < k < max(better) and k not in better]
print(f"SVD beats popularity at {len(better)} of {len(held_out_scores)} feature counts, from k = {min(better)} to k = {max(better)}{'' if not gaps else ' (not at ' + str(gaps) + ')'}")
first_below = next(int(k) for k in held_out_scores.index if k > best_k and held_out_scores.loc[k, "precision@10"] < pop_precision)
print(f"First feature count above the best where SVD falls below popularity: k = {first_below}")
print(f"Highest held-out accuracy at any k: {held_out_scores['accuracy'].max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(held_out_scores.index, held_out_scores["accuracy"], label="SVD")
axes[0].axhline(1 - held_out.mean(), color="grey", linestyle="--", label="predict no interaction")
axes[0].set_xlabel("Number of latent features")
axes[0].set_title("Held-out accuracy")
axes[0].legend()
axes[1].plot(held_out_scores.index, held_out_scores["precision@10"], label="SVD")
axes[1].axhline(pop_hits.mean() / 10, color="grey", linestyle="--", label="popularity")
axes[1].set_xlabel("Number of latent features")
axes[1].set_title("Held-out precision@10, new articles only")
axes[1].legend()
plt.show()

Held-out cells: 11480; ones: 218; new articles read by the 20 users: 191
Accuracy of predicting no interaction anywhere: 0.9810
Popularity: precision@10 0.080, recall@10 0.084
     accuracy  precision@10  recall@10
k                                     
10     0.9784         0.070     0.0733
50     0.9753         0.080     0.0838
110    0.9705         0.140     0.1466
150    0.9687         0.130     0.1361
190    0.9673         0.150     0.1571
210    0.9666         0.155     0.1623
310    0.9648         0.130     0.1361
410    0.9645         0.065     0.0681
510    0.9645         0.040     0.0419
690    0.9645         0.010     0.0105
Best held-out precision@10: k = 250 (0.170)
SVD beats popularity at 17 of 35 feature counts, from k = 70 to k = 390
First feature count above the best where SVD falls below popularity: k = 410
Highest held-out accuracy at any k: 0.9784


/var/folders/mq/m00xc0l165lcfkh2tz919jdh0000gn/T/ipykernel_6825/1162649384.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**How sure is that difference?** There are only 20 users to score, so a few lucky hits move the precision a lot. The cell below resamples the 20 users with replacement 5,000 times and reports the middle 95% of the difference in precision@10 between SVD with 200 features and popularity. Picking the best k on these same 20 users would flatter it, so the interval is for 200, the number chosen from the in-sample plot before this section was run.

In [68]:
svd_200_hits = hits_at_10(u_train[rows, :200] @ vt_train[:200][:, cols])
rng = np.random.default_rng(42)
draws = rng.integers(0, len(predictable_users), size=(5000, len(predictable_users)))
differences = (svd_200_hits[draws].mean(axis=1) - pop_hits[draws].mean(axis=1)) / 10
low, high = np.percentile(differences, [2.5, 97.5])
print(f"SVD k=200 hits: {int(svd_200_hits.sum())} of {10 * len(predictable_users)} picks; popularity hits: {int(pop_hits.sum())}")
print(f"Difference in precision@10 (SVD 200 minus popularity): {(svd_200_hits.mean() - pop_hits.mean()) / 10:.3f}, 95% interval {low:.3f} to {high:.3f}")
print(f"Users where SVD 200 found more, fewer, the same: {int((svd_200_hits > pop_hits).sum())}, {int((svd_200_hits < pop_hits).sum())}, {int((svd_200_hits == pop_hits).sum())}")

SVD k=200 hits: 33 of 200 picks; popularity hits: 16
Difference in precision@10 (SVD 200 minus popularity): 0.085, 95% interval 0.035 to 0.140
Users where SVD 200 found more, fewer, the same: 11, 2, 7


**What the held-out test shows.**

* Accuracy is the wrong measure here too. On held-out interactions it only falls as features are added, and its highest value (printed above) is below the 0.9810 of predicting no interaction at all. Rounding each score to 0 or 1 throws away the ranking, and the ranking is what a recommender is for.
* Ranked, the picture is different. SVD's ten picks per user beat popularity over the middle of the range of feature counts printed above, with the best precision@10 at 250 features (0.170 against popularity's 0.080). Past the point printed above, it falls below popularity: the extra features fit single users' training histories rather than taste that users share.
* At the 200 features chosen earlier, SVD found 33 new reads in 200 picks against popularity's 16. The 95% interval for the difference, 0.035 to 0.140, stays above zero, and SVD did better for 11 of the 20 users and worse for 2.
* The evidence is thin: 20 users and 191 new reads. And 662 of the 682 test users cannot be scored by SVD at all. For them, and for any brand-new user, recommendations still have to come from popularity or content.

So the held-out data supports keeping about 200 latent features, inside the range where SVD beats popularity, and it shows that the in-sample plot's message (more features are always better) is wrong once the model is asked about interactions it has not seen.

<a id='conclusions'></a>
### Extras
Using your workbook, you could now save your recommendations for each user, develop a class to make new predictions and update your results, and make a flask app to deploy your results.  These tasks are beyond what is required for this project.  However, from what you learned in the lessons, you certainly capable of taking these tasks on to improve upon your work here!


## Conclusion

> Congratulations!  You have reached the end of the Recommendation Systems project! 

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the rubric (see the **Project Rubric** page in the classroom). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.


## Directions to Submit

> Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should get a return code of 0, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations! 

In [69]:
import sys
from subprocess import call
call([sys.executable, '-m', 'nbconvert', '--to', 'html', 'Recommendations_with_IBM.ipynb'])

[NbConvertApp] Converting notebook Recommendations_with_IBM.ipynb to html


[NbConvertApp] Writing 592028 bytes to Recommendations_with_IBM.html


0